# Extension 4: Cohort Retention Forecasting

**Goal:** Given the first two weeks of a new cohort's retention data, predict their
week-8 and week-12 retention before those dates arrive.

**Models compared:**
- `LinearRegression` — baseline; fast and interpretable
- `GBTRegressor` — gradient-boosted trees; expected to capture non-linear decay patterns

**Data source:** `cohort_retention` (cohort-level weekly retention rates)

**Feature engineering:** Each training sample is one cohort. Features are early-week
retention rates (weeks 0–4); the target is a later-week retention rate (week 8 or 12).

---
**Prerequisites:** Run `make run-jobs` before opening this notebook.  
The `cohort_retention` table must have ≥ 1 cohort with data through week 12.

## Cell 1 — Setup

In [ ]:
import os
import pandas as pd
import psycopg2
from pyspark.sql import SparkSession

PG = dict(
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    dbname=os.getenv("POSTGRES_DB", "analytics"),
    user=os.getenv("POSTGRES_USER", "analytics_user"),
    password=os.getenv("POSTGRES_PASSWORD", "analytics_pass"),
)

def pg_query(sql: str) -> pd.DataFrame:
    """Execute SQL and return a pandas DataFrame."""
    # psycopg2's context manager only manages transactions (commit/rollback),
    # NOT the connection lifecycle — close() must be called explicitly.
    conn = psycopg2.connect(**PG)
    try:
        return pd.read_sql(sql, conn)
    finally:
        conn.close()

spark = (
    SparkSession.builder
    .appName("ML Feasibility — Retention Forecasting")
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", "goodnote-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "master:", spark.sparkContext.master)

## Cell 2 — Load Cohort Retention Data

In [ ]:
cohort_pd = pg_query("""
    SELECT cohort_week, week_number, cohort_size, retention_rate
    FROM cohort_retention
    ORDER BY cohort_week, week_number
""")

print(f"Rows loaded       : {len(cohort_pd):,}")
print(f"Cohorts           : {cohort_pd['cohort_week'].nunique()}")
print(f"Max week observed : {cohort_pd['week_number'].max()}")
cohort_pd.head(8)

## Cell 3 — Feature Engineering

Pivot the long-format cohort table into wide format so each row is one cohort and
each column is a weekly retention rate. Early weeks become features; a later week
becomes the regression target.

In [ ]:
pivot = (
    cohort_pd
    .pivot_table(index="cohort_week", columns="week_number", values="retention_rate")
    .reset_index()
)
pivot.columns = [f"week_{c}" if not isinstance(c, str) else c for c in pivot.columns]

# Cast cohort_week to str on both sides before merge — pivot_table may change the dtype
sizes = cohort_pd[cohort_pd["week_number"] == 0][["cohort_week", "cohort_size"]].copy()
pivot["cohort_week"] = pivot["cohort_week"].astype(str)
sizes["cohort_week"]  = sizes["cohort_week"].astype(str)
pivot = pivot.merge(sizes, on="cohort_week")

assert (pivot["cohort_size"] > 0).all(), \
    "One or more cohorts have cohort_size <= 0. Check the cohort_retention table."

print(f"Pivoted shape: {pivot.shape}")
print(f"Cohort size range: min={pivot['cohort_size'].min():.0f}, max={pivot['cohort_size'].max():.0f}")
print("Columns:", list(pivot.columns))
pivot.head()

### Layman Explanation

The raw cohort data is stored in **long format**: each row is one cohort at one week, like entries in a diary. But ML models want **wide format**: one row per cohort with each week's retention as its own column, like a spreadsheet where weeks are column headers.

`pivot_table` does this rotation automatically. We then attach each cohort's original size (from week 0) so the model can use it as a feature. Both tables are cast to the same type before merging to prevent a silent "no rows matched" failure.

### Technical Discussion

`pivot_table(index="cohort_week", columns="week_number", values="retention_rate")` rotates the long table into a matrix where:
- Rows = cohorts (indexed by `cohort_week`)
- Columns = week numbers (0, 1, 2, … 12)
- Values = `retention_rate`

Cohorts with missing data for a given week produce NaN in that cell — this is handled later by `dropna`.

Column names are renamed from integer week numbers to `"week_N"` strings for readability and to avoid issues with downstream Spark schema inference (Spark column names cannot be raw integers).

`astype(str)` on `cohort_week` before the merge is a defensive alignment step: `pivot_table` may convert a datetime index to `Timestamp` while the `sizes` DataFrame retains it as `object`. Even a minor dtype difference causes `merge` to produce zero rows silently.

### Terminology

| Term | Meaning |
|------|---------|
| **Long format (tidy data)** | One row per observation-variable combination. Each row is one cohort at one week. Easy to filter and aggregate but not directly usable as an ML feature matrix. |
| **Wide format** | One row per entity (cohort), one column per variable (week). Required for ML feature matrices. |
| **Pivot table** | A reshape operation that turns row values into column headers. Rows become the index; column values become column names. |
| **Cohort** | A group of users who started using the product in the same time window (here a calendar week). Cohort analysis compares how different groups retain over time. |
| **Retention rate** | The fraction of a cohort still active at a given week. `week_0 = 1.0` (all users are active on day 0); later weeks show the fraction who survived. |
| **dtype alignment** | Ensuring that join keys have the same data type on both sides of a merge, preventing silent mismatches. |
| **Merge key** | The column(s) used to match rows across two DataFrames in a join. |

## Cell 4 — Check Data Availability

We need at least some cohorts with data through week 12. If week 12 is not available,
fall back to forecasting week 8.

In [ ]:
available_weeks = [c for c in pivot.columns if c.startswith("week_")]
print("Available week columns:", sorted(available_weeks, key=lambda c: int(c.split("_")[1])))

if "week_12" in pivot.columns:
    TARGET = "week_12"
elif "week_8" in pivot.columns:
    TARGET = "week_8"
else:
    raise ValueError(
        f"Neither 'week_12' nor 'week_8' found in pivot columns: {sorted(available_weeks, key=lambda c: int(c.split("_")[1]))}. "
        "Run `make run-jobs` to populate cohort_retention with enough history."
    )

FEATURE_COLS = [c for c in ["week_0", "week_1", "week_2", "week_4", "cohort_size"] if c in pivot.columns]

# Validate that the minimum required early-week features are present
required_features = ["week_0", "week_1"]
missing_required = [c for c in required_features if c not in FEATURE_COLS]
if missing_required:
    raise ValueError(
        f"Required feature columns missing from dataset: {missing_required}. "
        "Cannot train without at least week_0 and week_1 retention data."
    )

print(f"\nTarget column : {TARGET}")
print(f"Feature cols  : {FEATURE_COLS}")

# Drop cohorts where any feature or target is missing
train_pivot = pivot.dropna(subset=FEATURE_COLS + [TARGET])
print(f"\nCohorts usable for training: {len(train_pivot)} of {len(pivot)}")

if len(train_pivot) < 4:
    print("WARNING: very few training samples. Generate more data with 'make generate-data'.")

### Layman Explanation

We select which weeks to use as **input features** and which week to **predict**. We use weeks 0–2 and 4 as inputs (early retention signal) and try to predict week 12 (long-term retention). Week 3 is intentionally skipped because it adds minimal information when week 2 and week 4 are already included — retention curves tend to be smooth.

If a cohort is missing any of the input weeks or the target week, it is dropped from training. We cannot teach the model from incomplete examples.

### Technical Discussion

`FEATURE_COLS` is built with a **guarded list comprehension**:
```python
[c for c in ["week_0", …] if c in pivot.columns]
```
This handles the case where a dataset does not have all weeks available (e.g. only 8 weeks of history) without raising a `KeyError`.

`TARGET` falls back from `"week_12"` to `"week_8"` if the dataset lacks week-12 data. This makes the notebook runnable on shorter datasets while still demonstrating the forecasting concept.

`dropna(subset=FEATURE_COLS + [TARGET])` silently removes cohorts with incomplete history. For cross-sectional cohort data (each cohort is an independent training sample) this is appropriate — it is not the same as time-series imputation. The dropped cohorts are simply too new to have accumulated the necessary history.

### Terminology

| Term | Meaning |
|------|---------|
| **Feature selection** | Choosing which input variables to give the model. More features is not always better — irrelevant or redundant features can hurt performance. |
| **Target variable** | The value the model is trained to predict. Here `week_12` retention (a number between 0 and 1). |
| **Fallback logic** | Code that uses a secondary option when the preferred one is unavailable. Robust code always considers what happens if data is missing. |
| **`dropna`** | Removes rows that have NaN in the specified columns. |
| **Cross-sectional data** | Each row is an independent observation (a cohort) rather than a time step in a sequence. This is different from time-series data where rows are correlated by order. |
| **Guarded list comprehension** | A list comprehension with an `if` condition that filters out elements — here, preventing `KeyError` when a column does not exist. |

## Cell 5 — MLlib Pipeline: LinearRegression vs GBTRegressor

In [ ]:
import math

from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")
evaluator = RegressionEvaluator(labelCol="label", metricName="rmse")
r2_eval   = RegressionEvaluator(labelCol="label", metricName="r2")

# Temporal split: train on the earliest 80% of cohorts, test on the most recent 20%.
# Random split would allow future cohorts into training, giving misleadingly low RMSE.
all_weeks = sorted(train_pivot["cohort_week"].unique())
n_train   = max(1, int(len(all_weeks) * 0.8))
train_weeks = set(all_weeks[:n_train])
test_weeks  = set(all_weeks[n_train:])

if len(test_weeks) < 2:
    print(f"WARNING: Test set has only {len(test_weeks)} cohort(s). "
          "Need >= 2 for meaningful evaluation. Run 'make generate-data' to add more cohorts.")

train = spark.createDataFrame(
    train_pivot[train_pivot["cohort_week"].isin(train_weeks)]
    [FEATURE_COLS + [TARGET, "cohort_week"]].rename(columns={TARGET: "label"})
)
train.cache()
test = spark.createDataFrame(
    train_pivot[train_pivot["cohort_week"].isin(test_weeks)]
    [FEATURE_COLS + [TARGET, "cohort_week"]].rename(columns={TARGET: "label"})
)
test.cache()
print(f"Train cohorts: {train.count()}   Test cohorts: {test.count()}")
print()

best_model    = None
best_rmse     = float("inf")
best_name     = ""
best_preds_pd = None  # test-set predictions for best model; reused in the scatter-plot cell

regressors = [
    ("LinearRegression", LinearRegression(featuresCol="features", labelCol="label")),
    ("GBTRegressor",     GBTRegressor(
        featuresCol="features", labelCol="label",
        maxIter=50, maxDepth=4, seed=42,
    )),
]

for name, regressor in regressors:
    pipe  = Pipeline(stages=[assembler, regressor])
    m     = pipe.fit(train)
    preds = m.transform(test)
    preds.cache()  # shared by both evaluators; avoids two separate scans of m.transform(test)
    rmse  = evaluator.evaluate(preds)
    r2    = r2_eval.evaluate(preds)
    print(f"{name:22s}  RMSE={rmse:.4f}  R²={r2:.4f}")
    if rmse < best_rmse:
        best_rmse     = rmse
        best_model    = m
        best_name     = name
        # Collect now while preds is cached; Cell 8 reuses this instead of re-running inference.
        best_preds_pd = preds.select("label", "prediction", "cohort_week").toPandas()
    preds.unpersist()

# Naive baseline: always predict the mean training-set label.
# Any useful model must outperform this.
train_labels  = train_pivot[train_pivot["cohort_week"].isin(train_weeks)][TARGET]
test_labels   = train_pivot[train_pivot["cohort_week"].isin(test_weeks)][TARGET]
mean_label    = train_labels.mean()
baseline_rmse = math.sqrt(((test_labels - mean_label) ** 2).mean())
print(f"{'Baseline (predict mean)':22s}  RMSE={baseline_rmse:.4f}  R²=0.0000  (reference)")
print(f"\nBest model: {best_name}  (RMSE={best_rmse:.4f})")
if best_rmse >= baseline_rmse:
    print("WARNING: Best model does not outperform the naive mean baseline. "
          "Consider adding more cohorts or richer features.")


### Layman Explanation

We train two models and compare them:

1. **Linear Regression** — assumes retention decays in a straight line based on the input features. Fast, easy to explain, but may miss curves and non-linear patterns.
2. **GBTRegressor** — a more powerful model that learns complex, non-linear patterns. Think of it as the "smarter but harder to explain" option.

Crucially, we split by **time**: older cohorts train the model, newer cohorts test it. This mimics real production use — we always predict the future from the past.

### Technical Discussion

The temporal split sorts cohorts chronologically and takes the earliest 80% for training and the most recent 20% for testing. This is essential for time-series–adjacent data:

- A **random split** would allow future cohorts (newer data) to inform the model when predicting past cohorts — a form of data leakage that inflates metrics.
- A **temporal split** ensures the model is evaluated on cohorts it genuinely could not have seen during training.

`LinearRegression` is the baseline: interpretable, no hyperparameters to tune, but assumes a linear relationship. `GBTRegressor(maxIter=50, maxDepth=4)` is the non-linear model: it builds 50 trees sequentially, each correcting the residuals of the previous ensemble.

RMSE (Root Mean Squared Error) is in the same units as `retention_rate` (fraction, 0–1). R² measures the proportion of variance explained; R²=1 is a perfect fit, R²=0 means the model is no better than predicting the mean.

### Terminology

| Term | Meaning |
|------|---------|
| **Temporal split** | Dividing a dataset by time — training on the past, testing on the future. Prevents data leakage in time-ordered data. |
| **Data leakage** | When information from the future (or test set) inadvertently influences the training process, producing misleadingly optimistic evaluation metrics. |
| **LinearRegression** | A model that fits a straight-line relationship between features and target: `ŷ = w₀ + w₁x₁ + w₂x₂ + …`. Fast and interpretable; cannot capture curves. |
| **GBTRegressor** | Gradient Boosted Tree Regressor. Builds an ensemble of trees where each tree fits the residuals of the previous ensemble. Captures non-linear patterns automatically. |
| **RMSE (Root Mean Squared Error)** | The square root of the average squared prediction error. In the same units as the target — here fractions of retention (0–1). Lower is better. |
| **R² (coefficient of determination)** | The fraction of target variance explained by the model. Range: −∞ to 1. R²=1 = perfect; R²=0 = predicts the mean; R²<0 = worse than the mean. |
| **Residuals** | The difference between actual and predicted values. Gradient boosting fits each new tree to the residuals of the current ensemble. |
| **Overfitting** | When a model memorises the training data and performs poorly on new data. `maxDepth=4` limits tree depth to reduce overfitting. |

## Cell 6 — Forecast Cohorts with Only Early-Week Data Available

This simulates the production use-case: new cohorts have data through week 4 only;
we predict their week-12 retention before it can be observed.

In [ ]:
new_cohorts_pd = pg_query("""
    SELECT cohort_week, week_number, cohort_size, retention_rate
    FROM cohort_retention
    WHERE week_number <= 4
""")

new_pivot = (
    new_cohorts_pd
    .pivot_table(index="cohort_week", columns="week_number", values="retention_rate")
    .reset_index()
)
new_pivot.columns = [f"week_{c}" if not isinstance(c, str) else c for c in new_pivot.columns]
new_pivot["cohort_week"] = new_pivot["cohort_week"].astype(str)
new_pivot = new_pivot.merge(
    new_cohorts_pd[new_cohorts_pd["week_number"] == 0][["cohort_week", "cohort_size"]]
    .assign(cohort_week=lambda d: d["cohort_week"].astype(str)),
    on="cohort_week",
)

# Keep only rows where all feature columns are available
new_pivot = new_pivot.dropna(subset=FEATURE_COLS)
print(f"Cohorts to forecast: {len(new_pivot)}")

if len(new_pivot) > 0:
    forecast_sdf = spark.createDataFrame(new_pivot[FEATURE_COLS + ["cohort_week"]])
    forecasts = best_model.transform(forecast_sdf)
    result = forecasts.select("cohort_week", "prediction").toPandas()
    result.columns = ["cohort_week", f"predicted_{TARGET}"]
    # Clip to valid retention range [0, 1] — regression can extrapolate outside.
    # Count clipped predictions: out-of-range values indicate the model is extrapolating,
    # which may signal insufficient training data or poor feature coverage.
    raw_preds = result[f"predicted_{TARGET}"].copy()
    result[f"predicted_{TARGET}"] = result[f"predicted_{TARGET}"].clip(0.0, 1.0).round(4)
    clipped = ((raw_preds < 0.0) | (raw_preds > 1.0)).sum()
    if clipped > 0:
        print(f"WARNING: {clipped} prediction(s) clipped to [0, 1] — "
              "model extrapolated outside valid retention bounds. "
              "Consider adding more training cohorts.")
    print(f"\nForecasted {TARGET} retention rates:")
    print(result.to_string(index=False))
else:
    print("No cohorts with complete early-week data to forecast.")

### Layman Explanation

Now we use the trained model to make real predictions on cohorts that only have data through week 4 — **exactly the situation we face in production** when a cohort is new. The model looks at early weeks and guesses where week-12 retention will land.

We also clip predictions to the range [0, 1]. Retention cannot be negative or above 100%, but regression models do not know that — they can mathematically produce numbers like 1.3 or −0.05. Clipping enforces the physical constraint.

### Technical Discussion

`best_model.transform(forecast_sdf)` runs the fitted pipeline in **inference mode**: the feature vector is assembled, and the model produces a `prediction` column (the estimated retention rate) without any training or label column present.

`.clip(0.0, 1.0)` is a hard constraint on the output domain. This is especially critical for `LinearRegression`, which has no built-in output constraints. GBT, being a tree-based model, is less likely to extrapolate wildly but can still produce slightly out-of-range values.

The same `pivot_table → rename → merge` logic from Cell 6 is repeated here for `new_cohorts_pd`. The `cohort_week` type alignment (`.astype(str)`) is applied to the `sizes` inline via `.assign(...)` — a pandas method that returns a modified copy without mutating the original.

### Terminology

| Term | Meaning |
|------|---------|
| **Inference** | Applying a trained model to new data to produce predictions. No training happens — only the forward pass through the pipeline. |
| **Output clipping** | Enforcing a hard lower and upper bound on model outputs. `clip(0.0, 1.0)` ensures predictions are valid retention rates. |
| **Domain constraint** | A physical or logical restriction on valid values. Retention rates are constrained to [0, 1] by definition. |
| **Extrapolation** | Predicting values outside the range seen during training. Models can extrapolate to nonsensical values; clipping is one defence. |
| **Production simulation** | Running the model in the same way it would operate in live production — predicting future outcomes from only the data available at that moment. |
| **`.assign()`** | A pandas method that returns a new DataFrame with a column added or modified, without mutating the original. Useful for concise, side-effect-free transformations. |

## Cell 7 — Retention Curve Visualisation

Plot the observed retention curves for each cohort to visually confirm the forecasting context.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

for cohort, group in cohort_pd.groupby("cohort_week"):
    ax.plot(
        group["week_number"],
        group["retention_rate"],
        alpha=0.5,
        linewidth=1,
        label=str(cohort),
    )

ax.set_xlabel("Weeks since cohort start")
ax.set_ylabel("Retention rate")
ax.set_title("Cohort Retention Curves")

# Only show legend if manageable number of cohorts
if cohort_pd["cohort_week"].nunique() <= 15:
    ax.legend(title="Cohort", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)

plt.tight_layout()
plt.show()

### Layman Explanation

Each line on this chart is the **life story of one cohort**: week 0 starts at 100% (all original users are active), and the line drops as users drift away over time. Steeper drops mean faster churn. Flatter lines mean stronger retention.

Plotting all cohorts together reveals whether the product is improving over time — if newer cohorts (later colours) show flatter curves than older ones, retention is getting better.

### Technical Discussion

`cohort_pd.groupby("cohort_week")` returns a grouped iterator. For each cohort, `group["week_number"]` is the x-axis and `group["retention_rate"]` is the y-axis.

`alpha=0.5` applies 50% transparency to each line, reducing **overplotting** when many cohorts overlap. Without transparency, overlapping lines become an unreadable solid mass.

The legend is conditionally shown only when `cohort_pd["cohort_week"].nunique() <= 15`. Beyond that threshold, the legend occupies too much space and individual cohort labels become illegible — omitting it is the pragmatic choice for exploratory analysis.

### Terminology

| Term | Meaning |
|------|---------|
| **Retention curve** | A line showing the fraction of a cohort still active at each time point. Starts at 1.0 and decreases monotonically (ideally). |
| **Cohort analysis** | Comparing how different groups of users (defined by acquisition week, region, etc.) behave over time — a key tool for understanding long-term engagement. |
| **Overplotting** | When many data points or lines overlap and obscure each other. Solutions include transparency (`alpha`), jitter, or aggregation. |
| **Alpha transparency** | A visual property (0 = fully transparent, 1 = fully opaque). Reducing alpha reveals density in overlapping regions. |
| **Survival rate** | Another term for retention rate — borrowed from survival analysis, where the "event" of interest is user churn rather than clinical death. |
| **`groupby` iterator** | In pandas, `groupby` returns an object that can be iterated over to produce (group_key, sub-DataFrame) pairs — one for each unique value of the grouping column. |

## Cell 8 — Predicted vs Actual (Test Set)

In [ ]:
if len(test_weeks) >= 2:
    test_preds = best_preds_pd  # reuse predictions collected during the training loop

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(test_preds["label"], test_preds["prediction"], alpha=0.7)

    lo = min(test_preds[["label", "prediction"]].min())
    hi = max(test_preds[["label", "prediction"]].max())
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="perfect forecast")

    for _, row in test_preds.iterrows():
        ax.annotate(str(row["cohort_week"]), (row["label"], row["prediction"]), fontsize=7)

    ax.set_xlabel(f"Actual {TARGET} retention")
    ax.set_ylabel(f"Predicted {TARGET} retention")
    ax.set_title(f"Predicted vs Actual — {best_name} (RMSE={best_rmse:.4f})")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"Test set has {len(test_weeks)} cohort(s) — need >= 2 for a meaningful scatter plot. "
          "Run 'make generate-data' to produce more cohorts.")


### Layman Explanation

Each dot on this scatter plot is one test cohort. Its **horizontal position** is the actual week-12 retention we observed; its **vertical position** is what the model predicted. If every prediction were perfect, all dots would line up exactly on the red diagonal line.

Dots above the line mean the model over-predicted retention (too optimistic). Dots below mean it under-predicted (too pessimistic). The spread of dots around the line gives an intuitive feel for the model's typical error — similar to what RMSE measures numerically.

### Technical Discussion

`best_model.transform(test)` applies the best pipeline to the held-out test cohorts. `.select("label", "prediction", "cohort_week").toPandas()` collects only the three columns needed for plotting, minimising driver memory usage.

The identity line (perfect forecast) is constructed from `lo = min(actual ∪ predicted)` to `hi = max(actual ∪ predicted)`. Using the combined range ensures the diagonal always spans the full plot area.

`ax.annotate(str(row["cohort_week"]), ...)` labels each dot with its cohort identifier, enabling specific cohorts to be investigated if their prediction error is large. For a dense test set this becomes unreadable — the annotations are most useful with 5–20 cohorts.

`test.count() > 0` guards against an empty test set (possible when fewer than 5 cohorts are available and the temporal split leaves none for testing). A more robust check would also verify `test.count() >= 2`.

### Terminology

| Term | Meaning |
|------|---------|
| **Scatter plot** | A chart where each observation is a point, positioned by two numeric variables (here: actual vs predicted retention). Used to visualise the relationship between two quantities. |
| **Identity line (diagonal)** | The line `y = x` on a predicted-vs-actual plot. A perfect model would have all points on this line. |
| **Prediction error** | The difference between predicted and actual values: `error = predicted − actual`. Positive = over-prediction; negative = under-prediction. |
| **Test set evaluation** | Measuring model performance on data the model never saw during training — the gold standard for estimating real-world performance. |
| **Annotation** | A text label placed next to a data point to identify it. Here each dot is labelled with its cohort week. |
| **Residual** | Synonymous with prediction error. Plotting residuals (y − ŷ) vs fitted values (ŷ) is a standard regression diagnostic for detecting bias. |
| **`toPandas()`** | Materialises a Spark DataFrame on the driver. Use sparingly — only on small result sets. |

## Cell 9 — Cleanup

In [ ]:
train.unpersist()
test.unpersist()
spark.stop()
print("Spark session stopped.")